# Compare FFC calculation methods

Investigates which per-FOV, per-pixel z-projection statistic -- **median**,
**max**, or **min** across the full z-stack -- and whether **Gaussian
smoothing** of the resulting field, best isolates real optical vignetting
from real tissue signal, on the same real dataset
(`LT066_sample_01/merfish`) and interior-FOV population
(`notebooks/tests/tissue_thickness/01_elevation_heatmap.ipynb` already
established: exterior/boundary FOVs are contaminated by real tissue, and
all 882 interior FOVs pooled -- not a curated subset -- gives a stable,
reproducible field).

**Why this needs its own investigation** (user's own prior experience,
stated directly): smoothing a raw per-pixel average is a blunt instrument
-- Gaussian-blurring the field changes the very spatial profile you're
trying to characterize, so "smoother-looking" is not the same as "more
correct." This notebook makes that tradeoff visible directly (smoothed vs.
unsmoothed profile plots side by side, per statistic) rather than assuming
smoothing helps.

**Method for each of the 3 x 2 = 6 combinations**: pool all 882 interior
FOVs' own per-pixel projection (median/max/min across their full 101-plane
z-stack, computed once per FOV via a SLURM array job -- reused across all
3 statistics since they're read from the same z-stack in one pass), mean
across FOVs, optionally Gaussian-smooth (`sigma=50px`, matching this
project's existing default), then percentile-normalize + floor-clip (same
recipe as `analysis.ffc.compute_ffc_field_for_color`'s own tail).

Self-contained/portable (`NOTEBOOK_GUIDELINES.md` #7): resolves the same
real dataset and interior-FOV population independently, rather than
depending on `01_elevation_heatmap.ipynb`'s own cache.

## 1 -- Setup

In [ ]:
%matplotlib inline
# %matplotlib widget  # uncomment for interactive pan/zoom (ipympl)

import os
import sys
import csv
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/),
# same convention as notebooks/before_imaging/regular/ (3 levels).
MERCI_DIR = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config import ExperimentConfig
from MERci.common.metadata import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.configs import get_fov_geometry, find_frame_table_for_hal_config
from MERci.acquisition.positions import find_exterior_fovs
from MERci.acquisition.merlin_config import load_microscope_orientation
from MERci.analysis.ffc import save_ffc_field, load_ffc_field
from MERci.visualization import get_merci_figures_dir

NOTEBOOK_NAME = "compare_ffc_methods"

PLOT_TITLE_FONTSIZE, PLOT_LABEL_FONTSIZE = 13, 11
PLOT_TICK_FONTSIZE, PLOT_LEGEND_FONTSIZE = 10, 10

## 2 -- Parameters

In [ ]:
# Same real dataset as notebooks/tests/tissue_thickness/01_elevation_heatmap.ipynb --
# see that notebook's own Parameters cell for why SAMPLE_DIR is set explicitly
# (too large to copy locally) rather than auto-detected.
SAMPLE_DIR = Path("/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/lineage_tracing/experiments/LT066_sample_01/merfish")

MICROSCOPE = "ST2"
OBJECTIVE  = "60X"
CHANNEL_NM = 405.0   # DAPI

STATISTICS = ["median", "max", "min"]
SMOOTH_SIGMA_PX          = 50.0   # this project's existing FFC default (0 = no smoothing)
NORMALIZE_PERCENTILE     = 99.99
FFC_MIN_VALUE            = 0.10

CACHE_DIR = SAMPLE_DIR / "analysis" / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "tests", NOTEBOOK_NAME, subfolder="calculate_ffc")

SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(SAMPLE_DIR / "MERci")
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)

pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE, OBJECTIVE)

config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt=SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix=".zarr", microscope=MICROSCOPE,
    pixel_size_um=pixel_size_um, image_size_px=image_size_px,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                image_suffix=config.image_suffix)

MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MICROSCOPE_ORIENTATION = load_microscope_orientation(MICROSCOPE, MICROSCOPE_ORIENTATION_DIR)

cells_round_id = meta.round_for_imaging_type("cells")
round_info = meta.rounds[cells_round_id]
positions = {fov_id: meta.fovs[fov_id].position
             for fov_id in round_info.fov_files if round_info.fov_files[fov_id]}

for s in meta.series_for_round(cells_round_id):
    if s.hal_config:
        ft_path = find_frame_table_for_hal_config(config.settings_dir / s.hal_config, config.metadata_dir)
        break
frame_table = pd.read_csv(ft_path, index_col=0)
channel_frames  = frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)].sort_values("z")
z_frame_indices = channel_frames.index.tolist()

boundary_fov_ids  = find_exterior_fovs(
    positions, config.step_size_um,
    connectivity=config.ffc_connectivity, tolerance_fraction=config.ffc_neighbor_tolerance,
)
interior_fov_ids = sorted(set(positions) - boundary_fov_ids)

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"{len(interior_fov_ids)} interior FOV(s) ({len(boundary_fov_ids)} boundary, excluded)")
print(f"{len(z_frame_indices)} z-plane(s) for {CHANNEL_NM} nm")
print(f"Cache  : {CACHE_DIR}")
print(f"Figures: {FIGURES_DIR}")

## 3 -- Per-FOV projections (median/max/min), all 882 interior FOVs

One SLURM array task per FOV, each reading its own full z-stack once and
writing all three requested statistics from that single read (`cli_
compute_fov_projections.py` + `cluster_submit.build_fov_projections_
array_script`). Re-run this cell later (after the job finishes) to pick up
newly-written results.

In [ ]:
proj_dir = CACHE_DIR / "fov_projections"
proj_dir.mkdir(parents=True, exist_ok=True)

def proj_path(fov_id, statistic):
    return proj_dir / f"fov{fov_id:04d}_{statistic}.npy"

def fov_is_ready(fov_id):
    return all(proj_path(fov_id, s).exists() for s in STATISTICS)

to_compute = [f for f in interior_fov_ids if not fov_is_ready(f)]
print(f"{len(interior_fov_ids) - len(to_compute)} / {len(interior_fov_ids)} interior FOV(s) "
      f"already fully cached; {len(to_compute)} more needed.")

USE_SLURM_ARRAY         = True   # set False to compute locally/serially instead (slow -- ~3h)
SLURM_ARRAY_CONCURRENCY = 50
SLURM_MEM               = "8gb"
SLURM_TIME              = "00:15:00"

if to_compute and USE_SLURM_ARRAY:
    from MERci.acquisition.cluster_submit import build_fov_projections_array_script, submit_sbatch, is_job_active

    job_sentinel = CACHE_DIR / "fov_projections_job.json"
    cached_job = json.loads(job_sentinel.read_text()) if job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute)
            and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = CACHE_DIR / "fov_projections_manifest.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path"])
            for fov_id in to_compute:
                writer.writerow([fov_id, round_info.fov_files[fov_id][0]])

        script_path = CACHE_DIR / "fov_projections.sh"
        build_fov_projections_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=proj_dir,
            frame_indices=z_frame_indices, statistics=STATISTICS, orientation=MICROSCOPE_ORIENTATION,
            n_pending=len(to_compute), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY, mem=SLURM_MEM, time=SLURM_TIME,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute:
    from MERci.common.io import read_image_frames
    from MERci.acquisition.merlin_config import apply_microscope_orientation
    from MERci.progress_display import ProgressReporter

    reporter = ProgressReporter(total=len(to_compute), label="Computing per-FOV projections (local)")
    for fov_id in reporter.wrap(to_compute):
        fpath = round_info.fov_files[fov_id][0]
        stack = read_image_frames(fpath, z_frame_indices).astype(np.float32)
        if "max" in STATISTICS:
            np.save(proj_path(fov_id, "max"), apply_microscope_orientation(np.max(stack, axis=0), **MICROSCOPE_ORIENTATION).astype(np.float32))
        if "min" in STATISTICS:
            np.save(proj_path(fov_id, "min"), apply_microscope_orientation(np.min(stack, axis=0), **MICROSCOPE_ORIENTATION).astype(np.float32))
        if "median" in STATISTICS:
            np.save(proj_path(fov_id, "median"), apply_microscope_orientation(np.median(stack, axis=0, overwrite_input=True), **MICROSCOPE_ORIENTATION).astype(np.float32))

## 4 -- Build the 6 field variants

Each of {median, max, min} x {smoothed (sigma=50px), unsmoothed}, streamed
one FOV at a time (never materializing all 882 images at once -- see
`01_elevation_heatmap.ipynb`'s own note on why).

In [ ]:
def build_ffc_field(paths, smooth_sigma_px, normalize_percentile, ffc_min_value):
    total = None
    for p in paths:
        img = np.load(p).astype(np.float64)
        total = img if total is None else total + img
    field = (total / len(paths)).astype(np.float32)
    if smooth_sigma_px and smooth_sigma_px > 0:
        field = gaussian_filter(field, sigma=smooth_sigma_px)
    norm_value = np.percentile(field, normalize_percentile)
    if norm_value > 0:
        field = field / norm_value
    return np.clip(field, ffc_min_value, None).astype(np.float32)

fov_ready = [f for f in interior_fov_ids if fov_is_ready(f)]
print(f"{len(fov_ready)} / {len(interior_fov_ids)} interior FOV(s) ready.")

fields = {}
if len(fov_ready) < len(interior_fov_ids):
    print("Still waiting on the SLURM array job (or local loop) above -- "
          "re-run both this cell and the previous one once every FOV is ready.")
else:
    for statistic in STATISTICS:
        paths = [proj_path(f, statistic) for f in interior_fov_ids]
        for smoothed, sigma, label in [(True, SMOOTH_SIGMA_PX, "smoothed"), (False, 0.0, "unsmoothed")]:
            key = f"{statistic}_{label}"
            cache_path = CACHE_DIR / f"ffc_field_{key}_{int(CHANNEL_NM)}nm.npz"
            if cache_path.exists():
                fields[key], _ = load_ffc_field(cache_path)
            else:
                fields[key] = build_ffc_field(paths, sigma, NORMALIZE_PERCENTILE, FFC_MIN_VALUE)
                save_ffc_field(cache_path, fields[key], {
                    "n_samples": len(paths), "statistic": statistic, "smooth_sigma_px": sigma,
                    "normalize_percentile": NORMALIZE_PERCENTILE, "ffc_min_value": FFC_MIN_VALUE,
                })
            print(f"{key:18s}: mean={fields[key].mean():.3f}  std={fields[key].std():.3f}  "
                  f"CV={fields[key].std() / fields[key].mean():.3f}")

## 5 -- Compare: 6-field grid

In [ ]:
if fields:
    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    for col, statistic in enumerate(STATISTICS):
        for row, label in enumerate(["smoothed", "unsmoothed"]):
            key = f"{statistic}_{label}"
            ax = axes[row, col]
            im = ax.imshow(fields[key], cmap="viridis", vmin=0, vmax=1.05)
            cv = fields[key].std() / fields[key].mean()
            ax.set_title(f"{statistic} -- {label}\nCV={cv:.3f}", fontsize=PLOT_TITLE_FONTSIZE)
            ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    fig.colorbar(im, ax=axes, shrink=0.7)
    fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.field_grid.png", dpi=150)
    plt.show()

## 6 -- Compare: 1D profiles, smoothed vs. unsmoothed overlaid per statistic

Directly shows what the Gaussian smoothing does to the profile shape --
the thing the user's own prior experience flagged as a real concern (a
"smoother" field is not automatically a more correct one if the blur has
altered the real vignette profile it's supposed to characterize).

In [ ]:
def field_profiles(field):
    h, w = field.shape
    return field[h // 2, :], field[:, w // 2], np.diagonal(field)

if fields:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
    for ax, statistic in zip(axes, STATISTICS):
        for label, style in [("smoothed", "-"), ("unsmoothed", "--")]:
            row_p, col_p, diag_p = field_profiles(fields[f"{statistic}_{label}"])
            ax.plot(row_p,  style, color="tab:blue",   label=f"{label} horizontal" if statistic == STATISTICS[0] else None)
            ax.plot(col_p,  style, color="tab:orange", label=f"{label} vertical"   if statistic == STATISTICS[0] else None)
            ax.plot(diag_p, style, color="tab:green",  label=f"{label} diagonal"   if statistic == STATISTICS[0] else None)
        ax.set_title(statistic, fontsize=PLOT_TITLE_FONTSIZE)
        ax.set_xlabel("pixel index along profile", fontsize=PLOT_LABEL_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    axes[0].set_ylabel("FFC field value", fontsize=PLOT_LABEL_FONTSIZE)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, fontsize=PLOT_LEGEND_FONTSIZE, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.05))
    fig.suptitle("Smoothed (solid) vs. unsmoothed (dashed) profiles, per statistic", fontsize=PLOT_TITLE_FONTSIZE)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.profiles_smoothed_vs_unsmoothed.png", dpi=150, bbox_inches="tight")
    plt.show()

## Discussion

**Smoothing (sigma=50px, at N=882 pooled FOVs) makes almost no difference**:
CV is nearly identical smoothed vs. unsmoothed for every statistic (median
0.221 vs 0.227, max 0.242 vs 0.250, min 0.185 vs 0.189), and the 1D
profiles are visually on top of each other -- smoothing only polishes off
high-frequency per-pixel jitter, it does not reshape the underlying curve.
Concretely: **smoothing does not rescue a contaminated field** -- median
and max's unsmoothed profiles already show a real, systematic bump around
pixel 1500-2000 on the diagonal (not just noise -- a genuine deviation
from a single-peaked, monotonically-falling vignette), and that same bump
survives smoothing essentially unchanged in the smoothed profiles. At this
sample size, smoothing is neither clearly harmful (as the user's prior
experience warned) nor clearly helpful here -- it's close to irrelevant,
because there isn't much residual per-pixel noise left for it to either
polish away cleanly or distort.

**The statistic choice is what actually matters, and MIN wins clearly**:
- **min** gives the cleanest field by a wide margin -- visibly the most
  even, single-peaked, radially-symmetric shape in the field grid, tightest
  jitter in its own *unsmoothed* profile (no smoothing needed to look
  clean), and no sign of the median/max bump.
- **median** is decent but shows real residual contamination (the
  diagonal-profile bump, and visible mottled/blob texture in the field
  grid at the same locations in both smoothed and unsmoothed versions).
- **max** is the most contaminated of the three (highest CV, same bump as
  median but more pronounced, most mottled field) -- makes physical sense:
  the max over 101 z-planes is close to guaranteed to catch a real nucleus
  wherever any passes through that pixel at all, so it's the statistic
  most exposed to real signal by construction.

This matches straightforward physical reasoning: a nucleus only occupies a
handful of a FOV's 101 z-planes at any given pixel, so the per-pixel
**minimum** across the full stack is overwhelmingly likely to be a pure
background reading -- much more robust to real tissue signal than the
median (which only needs a MINORITY of z-planes to be background to
resist contamination, but can still be pulled up if signal occupies more
than half the stack at some location) or the max (systematically pulled
up by real signal wherever it's present at all).

**Recommendation**: use **min** projection (smoothing optional -- it barely
changes min's already-clean result) for FFC field construction on this
kind of z-stacked data, rather than median. Not yet applied back to
`01_elevation_heatmap.ipynb` (which used median) -- that's the user's call
for a follow-up, not assumed here.